# NVIDIA BNR - Python Notebook

This notebook demonstrates how to use the NVIDIA Background Noise Removal (BNR) service with Python. NVIDIA BNR is a background noise removal model which removes unwanted noises from audio which improves speech intelligibility.

## Overview

The service processes audio files (WAV format) to output denoised audio with reduced background noise:

- **Service Integration**: Connect to BNR services via gRPC
- **Noise Removal**: AI-powered background noise removal
- **Model Versions**: Support for v1 and v2 models
- **Streaming Support**: Process audio in both streaming and transactional modes
- **Flexible Sample Rates**: Support for 16kHz and 48kHz audio
- **Intensity Control**: Seamless noise control with intensity ratio

## Requirements

- **Input Audio**: WAV files at 16kHz or 48kHz sample rate
- **Output**: Denoised WAV audio files
- **Service**: Access to a running BNR NIM service instance
- **Sample Rates**:
  - `48000`: 48kHz
  - `16000`: 16kHz


## Installation

### Requirements:
- Python 3.10 or above
- pip package manager
- gRPC dependencies from the requirements.txt file


In [ ]:
!pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your BNR service. The service can be running on your machine or on a remote server accessible from your environment.
To run the BNR NIM follow the instructions in [Getting Started](https://docs.nvidia.com/nim/maxine/bnr/latest/index.html).


In [ ]:
import os
import sys
import pathlib

# Setup paths for BNR modules
SCRIPT_PATH = str(pathlib.Path().resolve())
print(SCRIPT_PATH)
sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces/bnr"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../"))

# Service connection configuration
SERVICE_HOST = "localhost"  # Update to your service host
SERVICE_PORT = 8001         # Update to your service port
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target configured: {SERVICE_TARGET}")
print("Python paths configured for BNR modules")


## Import Libraries

Import the required libraries for gRPC communication, audio processing, and the BNR protobuf modules.

In [ ]:
import grpc
import time
import soundfile as sf
import numpy as np
from typing import Iterator, Optional
from tqdm.auto import tqdm
from IPython.display import Audio, display

# Import BNR modules
import bnr_pb2
import bnr_pb2_grpc

# Sample rate constants
CONST_SAMPLE_48KHZ = 48000
CONST_SAMPLE_16KHZ = 16000

print("All libraries imported successfully!")


## Helper Functions

Define functions for processing audio data and communicating with the BNR service. These functions handle the gRPC streaming protocol used by the BNR service.

### Function Overview

The BNR service uses gRPC bidirectional streaming for audio enhancement. The implementation consists of two main functions:

1. **Request Generator**: A Python iterator that yields request chunks to stream to the service
2. **Response Processor**: A function that processes the incoming gRPC data stream and writes the output file

### Streaming Protocol Details

- **Streaming Mode (recommended)**: Audio is split into 10ms chunks
- **Transactional Mode**: Entire audio file is sent in 64KB chunks
- **Intensity Ratio**: Optional parameter (0.0-1.0) to control denoising strength (default 1.0)
- **Response Stream**: Denoised audio data chunks


In [ ]:
def generate_request_for_inference(
    input_filepath: str,
    sample_rate: int,
    streaming: bool,
    intensity_ratio: Optional[float] = None,
) -> Iterator[bnr_pb2.EnhanceAudioRequest]:
    """Generate streaming requests for the BNR service.

    This function creates a Python iterator that yields gRPC request messages for streaming
    to the BNR service. The chunking strategy depends on the mode.

    Args:
        input_filepath: Path to the input WAV audio file
        sample_rate: Audio sample rate (16000 or 48000)
        streaming: If True, use streaming mode; if False, use transactional mode
        intensity_ratio: Optional denoising intensity (0.0 to 1.0)

    Yields:
        EnhanceAudioRequest messages containing audio data chunks
    """
    if not os.path.exists(input_filepath):
        raise FileNotFoundError(f"Audio file not found: {input_filepath}")

    # Send config first if intensity_ratio is specified
    if intensity_ratio is not None:
        config_request = bnr_pb2.EnhanceAudioRequest(
            config=bnr_pb2.EnhanceAudioConfig(intensity_ratio=intensity_ratio)
        )
        print(f"Sending intensity_ratio config: {intensity_ratio}")
        yield config_request

    if streaming:
        input_audio, sample_rate_file = sf.read(input_filepath)
        input_audio = input_audio.astype(np.float32)

        if sample_rate_file != sample_rate:
            raise ValueError(
                f"Sample rate mismatch: expected {sample_rate}, got {sample_rate_file}"
            )

        input_size_in_ms = 10
        samples_per_ms = sample_rate // 1000
        input_float_size = int(input_size_in_ms * samples_per_ms)

        pad_length = (input_float_size - len(input_audio) % input_float_size) % input_float_size
        if pad_length > 0:
            input_audio = np.pad(input_audio, (0, pad_length), "constant")

        print(f"Streaming mode: Chunk size = {input_size_in_ms}ms ({input_float_size} samples)")
        print(f"Total audio length: {len(input_audio)} samples")
        print(f"Number of chunks: {len(input_audio) // input_float_size}")

        for i in range(0, len(input_audio), input_float_size):
            data = input_audio[i : i + input_float_size]
            yield bnr_pb2.EnhanceAudioRequest(audio_stream_data=data.tobytes())
    else:
        DATA_CHUNKS = 64 * 1024  # bytes
        input_info = sf.info(input_filepath)
        sample_rate_file = input_info.samplerate
        if sample_rate_file != sample_rate:
            raise ValueError(
                f"Sample rate mismatch: expected {sample_rate}, got {sample_rate_file}"
            )

        file_size = os.path.getsize(input_filepath)

        print(f"Transactional mode: Sending audio in {DATA_CHUNKS} byte chunks")
        print(f"Total file size: {file_size / 1024:.1f} KB")

        with open(input_filepath, "rb") as fd:
            while True:
                buffer = fd.read(DATA_CHUNKS)
                if buffer == b"":
                    break
                yield bnr_pb2.EnhanceAudioRequest(audio_stream_data=buffer)


def write_output_file_from_response(
    response_iter: Iterator[bnr_pb2.EnhanceAudioResponse],
    output_filepath: str,
    sample_rate: int,
    streaming: bool,
) -> int:
    """Write the output file from the incoming gRPC data stream.

    Args:
        response_iter: Responses from the server to write into output file
        output_filepath: Path to output file
        sample_rate: Audio sample rate for output file
        streaming: If True, collect float32 audio data; if False, write raw WAV data

    Returns:
        Number of responses processed (only in streaming mode)
    """
    print(f"Writing output to {output_filepath}")

    if streaming:
        output_audio = []
        response_count = 0

        with tqdm(desc="Receiving audio chunks", unit="chunks", leave=False) as pbar:
            for response in response_iter:
                if response.HasField("audio_stream_data"):
                    response_count += 1
                    output_audio.append(np.frombuffer(response.audio_stream_data, np.float32))
                    pbar.update(1)

        combined_audio = np.hstack(output_audio)
        sf.write(output_filepath, combined_audio, sample_rate)

        print(f"Completed: Received {response_count} chunks")
        print(f"Output duration: {len(combined_audio) / sample_rate:.2f} seconds")

        return response_count
    else:
        total_bytes = 0

        with open(output_filepath, "wb") as fd:
            with tqdm(desc="Receiving audio data", unit="B", unit_scale=True, leave=False) as pbar:
                for response in response_iter:
                    if response.HasField("audio_stream_data"):
                        chunk_data = response.audio_stream_data
                        fd.write(chunk_data)
                        total_bytes += len(chunk_data)
                        pbar.update(len(chunk_data))

        print(f"Completed: Received {total_bytes / 1024:.1f} KB total")

        return 0


## Audio Processing

Process audio files with the BNR service. This method runs the background noise removal effect.


In [ ]:
def process_bnr_audio(
    input_path: str,
    output_path: str,
    sample_rate: int = CONST_SAMPLE_48KHZ,
    streaming: bool = True,
    intensity_ratio: Optional[float] = None,
) -> bool:
    """Process audio with BNR service.

    Args:
        input_path: Path to input audio file (WAV format)
        output_path: Path for output audio file
        sample_rate: Audio sample rate (16000 or 48000)
        streaming: If True, use streaming mode; if False, use transactional mode
        intensity_ratio: Optional denoising intensity (0.0 to 1.0)

    Returns:
        True if processing succeeded, False otherwise
    """
    try:
        print(f"\nProcessing audio: {input_path}")
        print(f"Sample rate: {sample_rate} Hz")
        print(f"Mode: {'Streaming' if streaming else 'Transactional'}")
        if intensity_ratio is not None:
            print(f"Intensity ratio: {intensity_ratio}")
        print(f"Connecting to service: {SERVICE_TARGET}")

        if not os.path.isfile(input_path):
            raise FileNotFoundError(f"Input file not found: {input_path}")

        input_info = sf.info(input_path)
        input_sample_rate = input_info.samplerate
        print(f"Input file sample rate: {input_sample_rate} Hz")

        if input_sample_rate != sample_rate:
            raise ValueError(
                f"Sample rate mismatch: expected {sample_rate} Hz, "
                f"but input file has {input_sample_rate} Hz"
            )

        with grpc.insecure_channel(SERVICE_TARGET) as channel:
            stub = bnr_pb2_grpc.BNRStub(channel)

            start_time = time.time()

            responses = stub.EnhanceAudio(
                generate_request_for_inference(
                    input_path, sample_rate, streaming, intensity_ratio
                )
            )

            response_count = write_output_file_from_response(
                responses, output_path, sample_rate, streaming
            )

            end_time = time.time()
            processing_time = end_time - start_time

            if streaming and response_count > 0:
                avg_latency = processing_time / response_count
                print(f"Average latency per chunk: {avg_latency*1000:.2f}ms")

            print(f"Processing complete in {processing_time:.2f}s")
            print(f"Output saved: {output_path}")

            return True

    except FileNotFoundError as e:
        print(f"Input file not found: {e}")
        print("   Please update file path with a valid audio file")
        return False
    except ValueError as e:
        print(f"Configuration error: {e}")
        return False
    except grpc.RpcError as e:
        print(f"Service connection failed: {e}")
        print(f"   Ensure the BNR NIM service is running at {SERVICE_TARGET}")
        return False
    except Exception as e:
        print(f"Processing failed: {e}")
        import traceback
        traceback.print_exc()
        return False


## Run BNR Processing - Basic Example

This cell demonstrates how to run BNR processing on an example audio file (.wav) using the streaming mode with the 48kHz sample rate.

**Important Note:** Ensure that your BNR NIM server is running with the matching sample rate and model version. The `--sample-rate` must match the model profile hosted on the server (e.g., `v2_48k` expects 48kHz input audio). Mismatches will result in processing errors.


In [ ]:
# Configuration
input_filepath = "../assets/bnr_48k_input.wav"    # Update with your input audio path
output_filepath = "bnr_48k_output.wav"             # Desired output audio path

# Process audio with settings - 48kHz, streaming mode
success = process_bnr_audio(
    input_path=input_filepath,
    output_path=output_filepath,
    sample_rate=CONST_SAMPLE_48KHZ,
    streaming=True
)

In [ ]:
# Playback: compare input and output audio
if success:
    print("Input audio (with noise):")
    display(Audio(filename=input_filepath))
    print("\nDenoised output audio:")
    display(Audio(filename=output_filepath))

## Advanced Usage Examples

Examples demonstrating different sample rates, processing modes, and intensity control.

### Modes and Options

BNR supports two processing modes and an optional intensity parameter:

| Mode | Description | Use Case |
|------|-------------|----------|
| **Streaming** | Audio processed in 10ms chunks | Real-time applications, live audio |
| **Transactional** | Entire file sent/received at once | Batch processing, offline audio cleanup |

| Option | Description |
|--------|-------------|
| **intensity_ratio** | Controls denoising strength (0.0 = no denoising, 1.0 = max denoising) |


### Example 1: Streaming Mode (48kHz)

Real-time processing with 10ms audio chunks.


In [ ]:
# Example 1: Streaming Mode (48kHz)
success = process_bnr_audio(
    input_path="../assets/bnr_48k_input.wav",
    output_path="output_48k_streaming.wav",
    sample_rate=CONST_SAMPLE_48KHZ,
    streaming=True
)

### Example 2: Transactional Mode (48kHz)

Standard processing for batch audio denoising.


In [ ]:
# Example 2: Transactional Mode (48kHz)
success = process_bnr_audio(
    input_path="../assets/bnr_48k_input.wav",
    output_path="output_48k_transactional.wav",
    sample_rate=CONST_SAMPLE_48KHZ,
    streaming=False
)

### Example 3: Intensity Control

Adjust denoising intensity using the `intensity_ratio` parameter. This is supported with both v1 and v2 model profiles.
- `1.0` = Maximum denoising (default)
- `0.5` = Moderate denoising
- `0.0` = No denoising effect


In [ ]:
# Example 3: Streaming with intensity control
success = process_bnr_audio(
    input_path="../assets/bnr_48k_input.wav",
    output_path="output_48k_intensity_0.5.wav",
    sample_rate=CONST_SAMPLE_48KHZ,
    streaming=True,
    intensity_ratio=0.5
)

### Example 4: 16kHz Model

For applications with 16kHz audio.

**Note:** The BNR NIM server must be running with a 16kHz model profile for this example.


In [ ]:
# Example 4: 16kHz Model
# Check if 16kHz input file exists, otherwise skip
input_16k = "../assets/bnr_16k_input.wav"
if os.path.exists(input_16k):
    success = process_bnr_audio(
        input_path=input_16k,
        output_path="output_16k.wav",
        sample_rate=CONST_SAMPLE_16KHZ,
        streaming=True
    )
else:
    print(f"16kHz input file not found at {input_16k}")
    print("Skipping 16kHz example. Provide a 16kHz WAV file to test this mode.")

For more information on BNR features, model configurations, and advanced usage, please refer to the [NVIDIA BNR Documentation](https://docs.nvidia.com/nim/maxine/bnr/latest/index.html).